# Part A — Big Data Analytics on DDoS Network Traffic

**Student:** Benjamine. S (258762A)
**Batch:** 18  
**Programme:** MSc Artificial Intelligence  
**University:** University of Moratuwa  
**Module:** Big Data Analytics Mini Project

---

## What This Notebook Does

This notebook analyses a real-world DDoS network traffic dataset using Apache Spark.
The dataset is the BCCC-cPacket-Cloud-DDoS-2024, published in March 2024 by York
University and cPacket Networks. It contains 540,494 network flow records with 319
features per flow, covering 17 DDoS attack types and 9 categories of benign traffic.

The analysis answers five questions about the data:

1. How are the attack types distributed — which are common and which are rare?
2. What are the measurable traffic signatures of each attack type?
3. How do timing patterns differ across attack types?
4. Which TCP flag combinations are characteristic of each attack?
5. Which features most clearly separate attack from benign traffic?

---

## Dataset

**Name:** BCCC-cPacket-Cloud-DDoS-2024  
**Source:** York University / cPacket Networks (March 2024)  
**Access:** Kaggle — dhoogla/bccc-cpacket-cloud-ddos-2024  
**License:** CC-BY-SA-4.0  
**Size:** 540,494 flows × 319 features  
**Format:** Parquet (29.5 MB compressed)

## Section 0 — Environment Setup

In [1]:
# Install PySpark — required in every new Colab session [not needed in VENV]
# !pip install pyspark --quiet

# Core PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F       # column functions: avg, count, round, when, etc.
from pyspark.sql.window import Window        # for window functions like RANK() OVER

# Visualisation — only called on small aggregated results after .toPandas()
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)

print("All libraries imported successfully.")

# --- Create SparkSession ---
spark = SparkSession.builder \
    .appName("BCCC_DDoS_PartA") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# --- Confirm it started ---
print(f"Spark version  : {spark.version}")
print(f"App name       : {spark.sparkContext.appName}")
print(f"Shuffle parts  : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver memory  : {spark.conf.get('spark.driver.memory')}")

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 17.0.19+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.19+0, mixed mode, sharing)

All libraries imported successfully.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 02:34:25 WARN Utils: Your hostname, Benjamines-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.66 instead (on interface en0)
26/05/12 02:34:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 02:34:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version  : 4.0.0
App name       : BCCC_DDoS_PartA
Shuffle parts  : 8
Driver memory  : 4g


## Section 1 — Data Loading

In [4]:
import os
import platform


HOME = os.path.expanduser("~")
KAGGLE_DIR = os.path.join(HOME, ".kaggle")
os.makedirs(KAGGLE_DIR, exist_ok=True)
print(f"Running on  : {platform.system()}")
print(f"Kaggle dir  : {KAGGLE_DIR}")

# --- Kaggle authentication ---
KAGGLE_TOKEN = os.environ.get('KAGGLE_TOKEN', '')

if not KAGGLE_TOKEN:
    print("\nWARNING: KAGGLE_TOKEN not set.")
    print("In terminal run: export KAGGLE_TOKEN=your_token_here")
    print("Then restart the notebook kernel.")
else:
    token_path = os.path.join(KAGGLE_DIR, "access_token")
    with open(token_path, 'w') as f:
        f.write(KAGGLE_TOKEN)
    os.chmod(token_path, 0o600)
    os.environ['KAGGLE_TOKEN'] = KAGGLE_TOKEN
    print("Kaggle credentials configured.")

    # --- Download dataset ---
    os.makedirs("./data", exist_ok=True)
    !kaggle datasets download \
        -d dhoogla/bccc-cpacket-cloud-ddos-2024 \
        --path ./data \
        --unzip --quiet

    DATA_PATH = "./data/bccc-cpacket-cloud-ddos-2024-merged.parquet"

    if os.path.exists(DATA_PATH):
        size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
        print(f"Dataset     : {DATA_PATH}")
        print(f"Size        : {size_mb:.1f} MB")

        # --- Load into Spark ---
        # Parquet embeds schema — no inferSchema scan needed
        df = spark.read.parquet(DATA_PATH)

        print(f"\nRows        : {df.count():,}")
        print(f"Columns     : {len(df.columns)}")

        # Register as SQL temp view for SparkSQL queries
        # This allows spark.sql("SELECT ... FROM ddos_raw")
        df.createOrReplaceTempView("ddos_raw")
        print("SQL view    : ddos_raw registered")
    else:
        print(f"File not found: {DATA_PATH}")
        print("Check that the download completed successfully.")

# Print schema to confirm all 319 features loaded with correct types
# df.printSchema()

Running on  : Darwin
Kaggle dir  : /Users/benjamine/.kaggle
Kaggle credentials configured.
Dataset URL: https://www.kaggle.com/datasets/dhoogla/bccc-cpacket-cloud-ddos-2024
License(s): CC-BY-SA-4.0
Dataset     : ./data/bccc-cpacket-cloud-ddos-2024-merged.parquet
Size        : 29.5 MB

Rows        : 540,494
Columns     : 319
SQL view    : ddos_raw registered


## Section 2 — Understanding the Label Structure

In [10]:
# --- label column: high-level category ---
print("=== LABEL column — high-level category ===")
spark.sql("""
    SELECT label,
           COUNT(*) AS flow_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM ddos_raw
    GROUP BY label
    ORDER BY flow_count DESC
""").show(truncate=False)

# --- activity column: specific traffic scenario ---
print("=== ACTIVITY column — specific traffic scenario ===")
spark.sql("""
    SELECT activity,
           label,
           COUNT(*) AS flow_count
    FROM ddos_raw
    GROUP BY activity, label
    ORDER BY label, flow_count DESC
""").show(40, truncate=False)

# --- Count distinct values in each ---
n_labels     = df.select("label").distinct().count()
n_activities = df.select("activity").distinct().count()

print(f"Distinct label values    : {n_labels}")
print(f"Distinct activity values : {n_activities}")
print()
print("Decision: all EDA sections group by 'activity'")
print("'label' is only used in Section 8 for binary Attack vs Benign comparison")

=== LABEL column — high-level category ===
+----------+----------+-----+
|label     |flow_count|pct  |
+----------+----------+-----+
|Benign    |349178    |64.60|
|Attack    |170436    |31.53|
|Suspicious|20880     |3.86 |
+----------+----------+-----+

=== ACTIVITY column — specific traffic scenario ===


26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 10:09:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/12 1

+--------------------------+----------+----------+
|activity                  |label     |flow_count|
+--------------------------+----------+----------+
|Attack-TCP-BYPass-V1      |Attack    |134110    |
|Attack-TCP-Flag-SYN       |Attack    |3147      |
|Attack-TCP-Flag-ACK       |Attack    |3135      |
|Attack-TCP-Flag-ACK-PSH   |Attack    |3109      |
|Attack-Killer-TCP         |Attack    |3046      |
|Attack-TCP-Valid-SYN      |Attack    |3003      |
|Attack-TCP-Flag-SYN-ACK   |Attack    |2990      |
|Attack-TCP-IGMP           |Attack    |2960      |
|Attack-TCP-Flag-MIX       |Attack    |2919      |
|Attack-Killall-v2         |Attack    |2824      |
|Attack-TCP-SYN            |Attack    |2739      |
|Attack-TCP-Control        |Attack    |2491      |
|Attack-TCP-Flag-SYN-TIME  |Attack    |980       |
|Attack-TCP-Flag-SYN-TFO   |Attack    |843       |
|Attack-TCP-Flag-OSYNP     |Attack    |758       |
|Attack-TCP-Flag-OSYN      |Attack    |757       |
|Attack-TCP-Flag-RST-ACK   |Att

### Finding — Label Structure

The dataset uses a two-level classification system. The `activity` column with
26 specific traffic scenarios is the correct grouping variable for all analysis.
Using `label` alone collapses 17 distinct attack types into one category.

**Class imbalance:** Attack-TCP-BYPass-V1 accounts for 134,110 flows — 78.6%
of all attack traffic. The remaining 16 attack types each contribute between
625 and 3,147 flows. This imbalance is acknowledged in the recommendation
system design — the hybrid approach combines co-occurrence signals (CF + ALS)
with feature-space similarity (content-based) so that minority attack types
with sparse interaction signals are still recommended based on their traffic
feature proximity to other attack types.